# Embedding-affinity fingerprint ("DNA" visualization; not crop composition or biological DNA)

This notebook asks whether intercropped-field pixels are simultaneously similar to geographically matched monocrop references in TESSERA's 128-dimensional representation. It keeps every selected 10 m pixel as an individual sample, validates reference prototypes with spatial-block out-of-fold monocrop fields, and summarizes matched-reference affinities at pixel and field level.

The four affinity scores are separately computed and non-compositional; they do not sum to one. They are representation-space similarities—not crop cover, abundance, plant count, biomass, yield, or a physical mixture percentage. Validation affinities and matched-reference mixture affinities are deliberately kept in separate tables and figures because their absolute scales come from different fitted references. An incomplete pipeline snapshot is always descriptive and watermarked, even if its internal validation thresholds pass.

In [ ]:
from pathlib import Path
import os
import sys

from IPython import get_ipython
from IPython.display import display

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "plain_tessera_incremental" / "__init__.py").is_file()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Run this notebook from inside the SpectraJam checkout.")
notebook_dir = repo_root / "plain_tessera_incremental" / "notebooks"
for path in (repo_root, notebook_dir):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from _pixel_analysis import (
    balanced_sample,
    clean_window_rows,
    l2_features,
    load_embeddings,
    load_static,
    scan_window,
)

OUTPUT_DIR = Path(
    os.environ.get(
        "TESSERA_OUTPUT_DIR",
        "/mnt/noobjam/harvard_tessera_incremental_v2",
    )
)
pipeline_complete = (OUTPUT_DIR / "COMPLETED.json").is_file()
TARGET_WINDOW = os.environ.get("TESSERA_WINDOW_ID", "w4")
MONO_CROPS = ["Maize", "Bean", "Irish Potato", "Rice"]
MIXTURE_PARENTS = {
    "Bean and Maize": ("Bean", "Maize"),
    "Irish Potato and Maize": ("Irish Potato", "Maize"),
}
ALL_LABELS = [*MONO_CROPS, *MIXTURE_PARENTS]
PIXEL_SIZE_M = 10
SPATIAL_BLOCK_M = 5000
MAX_VALIDATION_FIELDS_PER_LABEL = 48
MAX_MIXTURE_FIELDS_PER_LABEL = 48
MAX_PIXELS_PER_FIELD = None  # Keep every clean pixel in each selected field.
MATCHED_FIELDS_PER_CROP = 12
PIXEL_DNA_FIELDS_PER_MIXTURE = 4
MAX_FOLDS = 5
SPATIAL_ASSIGNMENT_ATTEMPTS = 10000
MIN_GATE_FIELDS_PER_PARENT = 12
MIN_GATE_BLOCKS_PER_PARENT = 3
MIN_GATE_RECALL = 0.5
MIN_GATE_WILSON_LOWER = 0.5
MAX_GATE_PERMUTATION_P = 0.05
N_LABEL_PERMUTATIONS = 2000
N_BOOTSTRAPS = 1000
MIN_MIXTURE_FIELDS_FOR_CI = 8
RANDOM_SEED = 20260708

print("Output:", OUTPUT_DIR)
print("DNA reference window:", TARGET_WINDOW)
print("Pipeline complete:", pipeline_complete)

In [ ]:
static = load_static(OUTPUT_DIR)
candidate_field_uids = set(
    static.fields.loc[
        static.fields["landcover"].isin(ALL_LABELS), "field_uid"
    ].astype(str)
)
if not candidate_field_uids:
    raise RuntimeError("None of the six requested DNA labels exists in fields.parquet.")
scan = scan_window(
    static,
    TARGET_WINDOW,
    retained_field_uids=candidate_field_uids,
)
clean = clean_window_rows(static, scan)
mono_reference_index, mono_reference_summary = balanced_sample(
    clean.rows,
    MONO_CROPS,
    max_fields_per_label=None,
    max_pixels_per_field=MAX_PIXELS_PER_FIELD,
    seed=RANDOM_SEED,
    min_fields_per_label=2,
)
mono_validation_index, mono_validation_summary = balanced_sample(
    clean.rows,
    MONO_CROPS,
    max_fields_per_label=MAX_VALIDATION_FIELDS_PER_LABEL,
    max_pixels_per_field=MAX_PIXELS_PER_FIELD,
    seed=RANDOM_SEED,
    min_fields_per_label=2,
)
mixture_index, mixture_summary = balanced_sample(
    clean.rows,
    list(MIXTURE_PARENTS),
    max_fields_per_label=MAX_MIXTURE_FIELDS_PER_LABEL,
    max_pixels_per_field=MAX_PIXELS_PER_FIELD,
    seed=RANDOM_SEED,
    min_fields_per_label=2,
)
selected_index = pd.concat(
    [mono_reference_index, mixture_index],
    ignore_index=True,
)
mono_validation_field_uids = set(
    mono_validation_index["field_uid"].astype(str).unique()
)
label_summary = pd.concat(
    {
        "all eligible monocrop references": mono_reference_summary,
        "monocrop validation subset": mono_validation_summary,
        "mixture analysis subset": mixture_summary,
    },
    names=["selection", "landcover"],
)

spatial_records = []
candidate_memberships = static.memberships[
    static.memberships["field_uid"].isin(candidate_field_uids)
]
for field_uid, rows in candidate_memberships.groupby("field_uid", sort=True):
    epsg_values = rows["utm_epsg"].astype(np.int64).unique()
    if len(epsg_values) != 1:
        raise RuntimeError(f"Field {field_uid} spans multiple UTM grids.")
    epsg = int(epsg_values[0])
    centroid_x_m = float(
        ((rows["pixel_x_index"].to_numpy(np.float64) + 0.5) * PIXEL_SIZE_M).mean()
    )
    centroid_y_m = float(
        ((rows["pixel_y_index"].to_numpy(np.float64) + 0.5) * PIXEL_SIZE_M).mean()
    )
    block_x = int(np.floor(centroid_x_m / SPATIAL_BLOCK_M))
    block_y = int(np.floor(centroid_y_m / SPATIAL_BLOCK_M))
    spatial_records.append(
        {
            "field_uid": str(field_uid),
            "utm_epsg": epsg,
            "centroid_x_m": centroid_x_m,
            "centroid_y_m": centroid_y_m,
            "centroid_longitude": float(rows["pixel_longitude"].mean()),
            "centroid_latitude": float(rows["pixel_latitude"].mean()),
            "spatial_block": f"epsg{epsg}-x{block_x}-y{block_y}",
        }
    )
field_spatial = pd.DataFrame(spatial_records)
if field_spatial["field_uid"].duplicated().any():
    raise RuntimeError("Spatial metadata contains duplicate field_uid values.")

pixels = load_embeddings(static, scan, selected_index).merge(
    field_spatial,
    on="field_uid",
    how="left",
    validate="many_to_one",
)
if pixels["spatial_block"].isna().any():
    raise RuntimeError("A selected field lacks 5 km spatial-block metadata.")
pixels["_feature"] = list(l2_features(pixels))

display(clean.diagnostics.rename("value").to_frame())
display(label_summary)
print(
    f"Candidate-filtered scan retained {len(scan.index):,} rows; loaded "
    f"{len(pixels):,} individual pixel vectors from "
    f"{pixels['field_uid'].nunique():,} completed fields."
)

In [ ]:
def training_weights(frame):
    field_pixel_counts = frame["field_uid"].value_counts().to_dict()
    field_labels = frame[["field_uid", "landcover"]].drop_duplicates()
    fields_per_crop = field_labels["landcover"].value_counts().to_dict()
    return np.array(
        [
            1.0 / (field_pixel_counts[field_uid] * fields_per_crop[label])
            for field_uid, label in frame[["field_uid", "landcover"]].itertuples(
                index=False, name=None
            )
        ],
        dtype=np.float64,
    )

def fit_reference(frame):
    labels = frame["landcover"].astype(str).to_numpy()
    missing = [crop for crop in MONO_CROPS if not np.any(labels == crop)]
    if missing:
        raise RuntimeError(f"Reference fit lacks monocrop classes: {missing}")
    x = np.stack(frame["_feature"].to_numpy())
    weights = training_weights(frame)
    mean = np.average(x, axis=0, weights=weights)
    variance = np.average((x - mean) ** 2, axis=0, weights=weights)
    scale = np.sqrt(np.maximum(variance, 0.0))
    scale[scale < 1e-8] = 1.0
    z = (x - mean) / scale
    prototypes = []
    distance_scales = []
    for crop in MONO_CROPS:
        mask = labels == crop
        prototype = np.average(z[mask], axis=0, weights=weights[mask])
        distances = np.linalg.norm(z[mask] - prototype, axis=1)
        distance_scale = np.sqrt(np.average(distances**2, weights=weights[mask]))
        prototypes.append(prototype)
        distance_scales.append(max(float(distance_scale), 1e-8))
    return {
        "mean": mean,
        "scale": scale,
        "prototypes": np.stack(prototypes),
        "distance_scales": np.asarray(distance_scales),
    }

def score_reference(model, frame):
    x = np.stack(frame["_feature"].to_numpy())
    z = (x - model["mean"]) / model["scale"]
    distances = np.linalg.norm(
        z[:, None, :] - model["prototypes"][None, :, :],
        axis=2,
    )
    affinities = np.exp(
        -0.5 * (distances / model["distance_scales"][None, :]) ** 2
    )
    return affinities, distances

def field_moments(frame):
    records = []
    for (field_uid, landcover, spatial_block), rows in frame.groupby(
        ["field_uid", "landcover", "spatial_block"],
        sort=True,
    ):
        x = np.stack(rows["_feature"].to_numpy())
        records.append(
            {
                "field_uid": str(field_uid),
                "landcover": str(landcover),
                "spatial_block": str(spatial_block),
                "pixel_samples": len(x),
                "feature_mean": x.mean(axis=0),
                "feature_second": (x * x).mean(axis=0),
            }
        )
    return pd.DataFrame(records)

def fit_moment_reference(frame, labels=None):
    labels = (
        frame["landcover"].astype(str).to_numpy()
        if labels is None
        else np.asarray(labels, dtype=object)
    )
    means = np.stack(frame["feature_mean"].to_numpy())
    seconds = np.stack(frame["feature_second"].to_numpy())
    class_means = []
    class_seconds = []
    for crop in MONO_CROPS:
        mask = labels == crop
        if not mask.any():
            raise RuntimeError(f"Moment reference lacks crop {crop}.")
        class_means.append(means[mask].mean(axis=0))
        class_seconds.append(seconds[mask].mean(axis=0))
    class_means = np.stack(class_means)
    class_seconds = np.stack(class_seconds)
    mean = class_means.mean(axis=0)
    second = class_seconds.mean(axis=0)
    scale = np.sqrt(np.maximum(second - mean * mean, 0.0))
    scale[scale < 1e-8] = 1.0
    prototypes = (class_means - mean) / scale
    class_z_second = (
        class_seconds - 2.0 * mean * class_means + mean * mean
    ) / (scale * scale)
    distance_scale_squared = np.maximum(
        (class_z_second - prototypes * prototypes).sum(axis=1),
        1e-16,
    )
    return {
        "mean": mean,
        "scale": scale,
        "prototypes": prototypes,
        "distance_scales": np.sqrt(distance_scale_squared),
    }

def score_field_moments(model, frame):
    means = np.stack(frame["feature_mean"].to_numpy())
    seconds = np.stack(frame["feature_second"].to_numpy())
    z_mean = (means - model["mean"]) / model["scale"]
    z_second = (
        seconds - 2.0 * model["mean"] * means + model["mean"] ** 2
    ) / (model["scale"] ** 2)
    distance_squared = (
        z_second[:, None, :].sum(axis=2)
        - 2.0 * z_mean @ model["prototypes"].T
        + (model["prototypes"] ** 2).sum(axis=1)[None, :]
    )
    distance_squared = np.maximum(distance_squared, 0.0)
    affinities = np.exp(
        -0.5
        * distance_squared
        / (model["distance_scales"][None, :] ** 2)
    )
    return affinities, np.sqrt(distance_squared)

def assign_spatial_folds(field_table):
    field_counts = (
        field_table.groupby("landcover")["field_uid"]
        .nunique()
        .reindex(MONO_CROPS, fill_value=0)
    )
    block_counts = (
        field_table.groupby("landcover")["spatial_block"]
        .nunique()
        .reindex(MONO_CROPS, fill_value=0)
    )
    if (
        (field_counts < MIN_GATE_FIELDS_PER_PARENT).any()
        or (block_counts < MIN_GATE_BLOCKS_PER_PARENT).any()
    ):
        return None, 0, field_counts, block_counts
    n_folds = min(MAX_FOLDS, int(block_counts.min()))
    blocks = np.array(sorted(field_table["spatial_block"].unique()), dtype=object)
    block_crop_counts = np.zeros((len(blocks), len(MONO_CROPS)), dtype=np.int64)
    block_index = {block: index for index, block in enumerate(blocks)}
    crop_index = {crop: index for index, crop in enumerate(MONO_CROPS)}
    for (block, crop), count in field_table.groupby(
        ["spatial_block", "landcover"]
    ).size().items():
        block_crop_counts[block_index[block], crop_index[crop]] = int(count)

    rng = np.random.default_rng(RANDOM_SEED + 1)
    best_assignment = None
    best_score = np.inf
    target = block_crop_counts.sum(axis=0) / n_folds
    for _ in range(SPATIAL_ASSIGNMENT_ATTEMPTS):
        assignment = rng.integers(0, n_folds, size=len(blocks))
        if len(np.unique(assignment)) != n_folds:
            continue
        fold_counts = np.stack(
            [block_crop_counts[assignment == fold].sum(axis=0) for fold in range(n_folds)]
        )
        if (fold_counts == 0).any():
            continue
        score = float((((fold_counts - target) ** 2) / (target + 1e-9)).sum())
        if score < best_score:
            best_score = score
            best_assignment = assignment.copy()
    if best_assignment is None:
        return None, 0, field_counts, block_counts
    return (
        dict(zip(blocks, best_assignment, strict=True)),
        n_folds,
        field_counts,
        block_counts,
    )

mono_reference = (
    pixels[pixels["landcover"].isin(MONO_CROPS)].copy().reset_index(drop=True)
)
mono = mono_reference[
    mono_reference["field_uid"].isin(mono_validation_field_uids)
].copy().reset_index(drop=True)
mixtures = pixels[pixels["landcover"].isin(MIXTURE_PARENTS)].copy().reset_index(drop=True)
mono_reference_fields = field_moments(mono_reference)
mono_fields = field_moments(mono)
block_to_fold, n_folds, mono_field_counts, mono_block_counts = assign_spatial_folds(
    mono_fields
)
spatial_validation_feasible = block_to_fold is not None
if spatial_validation_feasible:
    mono_fields["fold"] = mono_fields["spatial_block"].map(block_to_fold).astype(int)
else:
    mono_fields["fold"] = -1

print(
    f"Monocrop validation fields={len(mono_fields)}, 5 km blocks="
    f"{mono_fields['spatial_block'].nunique()}, spatial folds={n_folds or 'unavailable'}."
)
print(
    f"Eligible geographic reference fields={len(mono_reference_fields)} "
    f"across {mono_reference_fields['spatial_block'].nunique()} blocks."
)

In [ ]:
def classification_metrics(actual, predicted):
    confusion = np.zeros((len(MONO_CROPS), len(MONO_CROPS)), dtype=np.float64)
    class_index = {crop: index for index, crop in enumerate(MONO_CROPS)}
    for actual_label, predicted_label in zip(actual, predicted, strict=True):
        confusion[class_index[actual_label], class_index[predicted_label]] += 1.0
    row_totals = confusion.sum(axis=1)
    row_normalized = np.divide(
        confusion,
        row_totals[:, None],
        out=np.zeros_like(confusion),
        where=row_totals[:, None] > 0,
    )
    recall = np.diag(row_normalized)
    return {
        "confusion": confusion,
        "row_normalized": row_normalized,
        "recall": recall,
        "balanced_accuracy": float(recall.mean()),
    }

def wilson_lower(successes, total, z=1.959963984540054):
    if total <= 0:
        return 0.0
    proportion = successes / total
    denominator = 1.0 + z * z / total
    center = proportion + z * z / (2.0 * total)
    adjustment = z * np.sqrt(
        proportion * (1.0 - proportion) / total + z * z / (4.0 * total * total)
    )
    return float((center - adjustment) / denominator)

def spatial_oof_predictions(labels):
    labels = np.asarray(labels, dtype=object)
    predictions = np.empty(len(mono_fields), dtype=object)
    affinities = np.empty((len(mono_fields), len(MONO_CROPS)), dtype=np.float64)
    for fold in range(n_folds):
        train_mask = mono_fields["fold"].ne(fold).to_numpy()
        test_mask = ~train_mask
        train_blocks = set(mono_fields.loc[train_mask, "spatial_block"])
        test_blocks = set(mono_fields.loc[test_mask, "spatial_block"])
        if train_blocks & test_blocks:
            raise RuntimeError("A 5 km spatial block leaked across validation folds.")
        model = fit_moment_reference(
            mono_fields.loc[train_mask],
            labels=labels[train_mask],
        )
        fold_affinities, _ = score_field_moments(
            model,
            mono_fields.loc[test_mask],
        )
        affinities[test_mask] = fold_affinities
        predictions[test_mask] = np.asarray(MONO_CROPS, dtype=object)[
            np.argmax(fold_affinities, axis=1)
        ]
    return predictions, affinities

true_field_labels = mono_fields["landcover"].astype(str).to_numpy()
if spatial_validation_feasible:
    oof_predictions, oof_field_affinities = spatial_oof_predictions(true_field_labels)
    validation_metrics = classification_metrics(true_field_labels, oof_predictions)
    own_affinity = np.array(
        [
            oof_field_affinities[index, MONO_CROPS.index(label)]
            for index, label in enumerate(true_field_labels)
        ]
    )
    max_off_target = np.array(
        [
            np.max(
                np.delete(
                    oof_field_affinities[index],
                    MONO_CROPS.index(label),
                )
            )
            for index, label in enumerate(true_field_labels)
        ]
    )
    own_beats = own_affinity > max_off_target

    permutation_rng = np.random.default_rng(RANDOM_SEED + 2)
    permutation_scores = np.empty(N_LABEL_PERMUTATIONS, dtype=np.float64)
    fold_values = mono_fields["fold"].to_numpy()
    for permutation in range(N_LABEL_PERMUTATIONS):
        permuted = true_field_labels.copy()
        for fold in range(n_folds):
            positions = np.flatnonzero(fold_values == fold)
            shuffled = true_field_labels[positions].copy()
            permutation_rng.shuffle(shuffled)
            permuted[positions] = shuffled
        permuted_predictions, _ = spatial_oof_predictions(permuted)
        permutation_scores[permutation] = classification_metrics(
            permuted,
            permuted_predictions,
        )["balanced_accuracy"]
    permutation_p_value = (
        1
        + np.count_nonzero(
            permutation_scores >= validation_metrics["balanced_accuracy"]
        )
    ) / (N_LABEL_PERMUTATIONS + 1)
else:
    oof_predictions = np.array([], dtype=object)
    oof_field_affinities = np.empty((0, len(MONO_CROPS)))
    own_affinity = np.array([])
    max_off_target = np.array([])
    own_beats = np.array([], dtype=bool)
    permutation_scores = np.array([])
    permutation_p_value = np.nan
    validation_metrics = {
        "confusion": np.zeros((len(MONO_CROPS), len(MONO_CROPS))),
        "row_normalized": np.zeros((len(MONO_CROPS), len(MONO_CROPS))),
        "recall": np.zeros(len(MONO_CROPS)),
        "balanced_accuracy": 0.0,
    }

gate_rows = []
for crop_index, crop in enumerate(MONO_CROPS):
    crop_mask = true_field_labels == crop
    successes = int(own_beats[crop_mask].sum()) if spatial_validation_feasible else 0
    total = int(crop_mask.sum())
    lower = wilson_lower(successes, total)
    recall = float(validation_metrics["recall"][crop_index])
    fields = int(mono_field_counts[crop])
    blocks = int(mono_block_counts[crop])
    gate_rows.append(
        {
            "parent": crop,
            "fields": fields,
            "spatial_blocks": blocks,
            "oof_recall": recall,
            "own_beats_off_target": successes / total if total else 0.0,
            "own_beats_wilson_lower": lower,
            "parent_gate_passed": (
                spatial_validation_feasible
                and fields >= MIN_GATE_FIELDS_PER_PARENT
                and blocks >= MIN_GATE_BLOCKS_PER_PARENT
                and recall >= MIN_GATE_RECALL
                and lower > MIN_GATE_WILSON_LOWER
            ),
        }
    )
parent_gate = pd.DataFrame(gate_rows).set_index("parent")
validation_gate_passed = bool(
    spatial_validation_feasible
    and parent_gate["parent_gate_passed"].all()
    and permutation_p_value <= MAX_GATE_PERMUTATION_P
)
reference_gate_passed = bool(pipeline_complete and validation_gate_passed)
interpretation_status = (
    "DESCRIPTIVE ONLY — PARTIAL PIPELINE OUTPUT"
    if not pipeline_complete
    else (
        "REFERENCE GATE PASSED — matched affinities remain descriptive, not proportions"
        if validation_gate_passed
        else "DESCRIPTIVE ONLY — REFERENCE GATE FAILED"
    )
)

figure, axes = plt.subplots(1, 2, figsize=(14, 5.5))
image = axes[0].imshow(
    validation_metrics["row_normalized"],
    vmin=0,
    vmax=1,
    cmap="Blues",
)
axes[0].set_xticks(range(len(MONO_CROPS)), MONO_CROPS, rotation=35, ha="right")
axes[0].set_yticks(range(len(MONO_CROPS)), MONO_CROPS)
axes[0].set_xlabel("highest-affinity monocrop reference")
axes[0].set_ylabel("held-out monocrop field")
axes[0].set_title(
    f"5 km spatial-block OOF validation\n"
    f"balanced accuracy={validation_metrics['balanced_accuracy']:.3f}"
)
for row in range(len(MONO_CROPS)):
    for column in range(len(MONO_CROPS)):
        axes[0].text(
            column,
            row,
            f"{validation_metrics['row_normalized'][row, column]:.2f}",
            ha="center",
            va="center",
        )
figure.colorbar(image, ax=axes[0], fraction=0.046, pad=0.04)

if spatial_validation_feasible:
    for crop in MONO_CROPS:
        mask = true_field_labels == crop
        axes[1].scatter(
            max_off_target[mask],
            own_affinity[mask],
            s=28,
            alpha=0.7,
            label=crop,
        )
    limit = max(axes[1].get_xlim()[1], axes[1].get_ylim()[1])
    axes[1].plot([0, limit], [0, limit], color="black", linestyle="--", linewidth=1)
    axes[1].set_xlim(0, limit)
    axes[1].set_ylim(0, limit)
    axes[1].legend(fontsize=8)
else:
    axes[1].text(
        0.5,
        0.5,
        "Spatial OOF unavailable:\nparent field/block minimums or fold assignment failed",
        ha="center",
        va="center",
        transform=axes[1].transAxes,
    )
axes[1].set_xlabel("largest off-target affinity")
axes[1].set_ylabel("correct-parent affinity")
axes[1].set_title(
    f"Field controls; permutation p="
    f"{permutation_p_value:.4f}" if np.isfinite(permutation_p_value)
    else "Field controls; permutation unavailable"
)
figure.suptitle(interpretation_status, color="darkgreen" if reference_gate_passed else "crimson")
figure.tight_layout()
plt.show()

display(parent_gate)
display(
    pd.Series(
        {
            "pipeline_complete": pipeline_complete,
            "spatial_validation_feasible": spatial_validation_feasible,
            "spatial_folds": n_folds,
            "field_balanced_accuracy": validation_metrics["balanced_accuracy"],
            f"label_permutation_p_{N_LABEL_PERMUTATIONS}": permutation_p_value,
            "validation_gate_passed": validation_gate_passed,
            "reference_gate_passed": reference_gate_passed,
        },
        name="validation",
    ).to_frame()
)
print(interpretation_status)

In [ ]:
def haversine_matrix_km(left_lon, left_lat, right_lon, right_lat):
    left_lon = np.deg2rad(np.asarray(left_lon, dtype=np.float64))[:, None]
    left_lat = np.deg2rad(np.asarray(left_lat, dtype=np.float64))[:, None]
    right_lon = np.deg2rad(np.asarray(right_lon, dtype=np.float64))[None, :]
    right_lat = np.deg2rad(np.asarray(right_lat, dtype=np.float64))[None, :]
    delta_lon = right_lon - left_lon
    delta_lat = right_lat - left_lat
    value = (
        np.sin(delta_lat / 2.0) ** 2
        + np.cos(left_lat) * np.cos(right_lat) * np.sin(delta_lon / 2.0) ** 2
    )
    return 6371.0088 * 2.0 * np.arcsin(np.sqrt(np.clip(value, 0.0, 1.0)))

def attach_mixture_scores(frame, model, mixture_label):
    result = frame.copy()
    affinities, _ = score_reference(model, result)
    for column, values in zip(affinity_columns, affinities.T, strict=True):
        result[column] = values
    parent_a, parent_b = MIXTURE_PARENTS[mixture_label]
    a = result[crop_to_column[parent_a]].to_numpy()
    b = result[crop_to_column[parent_b]].to_numpy()
    off_columns = [
        crop_to_column[crop]
        for crop in MONO_CROPS
        if crop not in {parent_a, parent_b}
    ]
    off_target = result[off_columns].max(axis=1).to_numpy()
    result["dual_target_gap"] = np.minimum(a, b) - off_target
    result["parent_contrast"] = (a - b) / (a + b + 1e-12)
    result["parent_a"] = parent_a
    result["parent_b"] = parent_b
    return result

def watermark(figure):
    if not reference_gate_passed:
        figure.text(
            0.5,
            0.5,
            interpretation_status,
            ha="center",
            va="center",
            fontsize=20,
            color="crimson",
            alpha=0.20,
            rotation=25,
            weight="bold",
        )

affinity_columns = [
    f"affinity_{crop.lower().replace(' ', '_')}" for crop in MONO_CROPS
]
crop_to_column = dict(zip(MONO_CROPS, affinity_columns, strict=True))
selected_field_geo = field_spatial[
    field_spatial["field_uid"].isin(pixels["field_uid"].unique())
].copy()
field_label_lookup = (
    pixels[["field_uid", "landcover"]].drop_duplicates().set_index("field_uid")["landcover"]
)
selected_field_geo["landcover"] = selected_field_geo["field_uid"].map(field_label_lookup)

matched_reference_fields = {}
matched_models = {}
match_distance_records = []
mixture_score_parts = []
for mixture_label in MIXTURE_PARENTS:
    mixture_rows = mixtures[mixtures["landcover"].eq(mixture_label)].copy()
    mixture_field_ids = sorted(mixture_rows["field_uid"].unique())
    mixture_geo = selected_field_geo[
        selected_field_geo["field_uid"].isin(mixture_field_ids)
    ]
    if mixture_geo.empty:
        raise RuntimeError(f"No spatial metadata for mixture {mixture_label}.")
    reference_by_crop = {}
    for crop in MONO_CROPS:
        candidates = selected_field_geo[
            selected_field_geo["landcover"].eq(crop)
        ].copy()
        if candidates.empty:
            raise RuntimeError(f"No monocrop reference candidates for {crop}.")
        distance_matrix = haversine_matrix_km(
            candidates["centroid_longitude"],
            candidates["centroid_latitude"],
            mixture_geo["centroid_longitude"],
            mixture_geo["centroid_latitude"],
        )
        candidates["match_distance_km"] = distance_matrix.min(axis=1)
        candidates = candidates.sort_values(
            ["match_distance_km", "field_uid"],
            kind="stable",
        )
        selected = candidates.head(MATCHED_FIELDS_PER_CROP)
        reference_by_crop[crop] = selected["field_uid"].astype(str).tolist()
        mixture_to_reference = haversine_matrix_km(
            mixture_geo["centroid_longitude"],
            mixture_geo["centroid_latitude"],
            selected["centroid_longitude"],
            selected["centroid_latitude"],
        ).min(axis=1)
        match_distance_records.append(
            {
                "mixture_label": mixture_label,
                "reference_crop": crop,
                "eligible_reference_fields": len(candidates),
                "reference_fields": len(selected),
                "mixture_fields": len(mixture_field_ids),
                "median_reference_to_mixture_km": float(
                    selected["match_distance_km"].median()
                ),
                "max_reference_to_mixture_km": float(
                    selected["match_distance_km"].max()
                ),
                "median_mixture_to_reference_km": float(
                    np.median(mixture_to_reference)
                ),
                "max_mixture_to_reference_km": float(
                    np.max(mixture_to_reference)
                ),
            }
        )
    matched_reference_fields[mixture_label] = reference_by_crop
    reference_ids = {
        field_uid
        for field_ids_for_crop in reference_by_crop.values()
        for field_uid in field_ids_for_crop
    }
    reference_rows = mono_reference[
        mono_reference["field_uid"].isin(reference_ids)
    ].copy()
    matched_model = fit_reference(reference_rows)
    matched_models[mixture_label] = matched_model
    scored = attach_mixture_scores(
        mixture_rows,
        matched_model,
        mixture_label,
    )
    scored["reference_cohort"] = mixture_label
    mixture_score_parts.append(scored)

match_summary = pd.DataFrame(match_distance_records)
match_summary["interpretation"] = interpretation_status
mixture_pixel_scores = pd.concat(mixture_score_parts, ignore_index=True)
field_fingerprints = mixture_pixel_scores.groupby(
    ["field_uid", "landcover"], as_index=False
).agg(
    **{column: (column, "median") for column in affinity_columns},
    pixel_samples=("pixel_id", "nunique"),
)

display(match_summary.set_index(["mixture_label", "reference_crop"]))

figure, axes = plt.subplots(
    1,
    len(MIXTURE_PARENTS),
    figsize=(7.0 * len(MIXTURE_PARENTS), 7),
    squeeze=False,
)
for axis, mixture_label in zip(axes[0], MIXTURE_PARENTS, strict=True):
    rows = field_fingerprints[
        field_fingerprints["landcover"].eq(mixture_label)
    ].sort_values("field_uid", kind="stable")
    matrix = rows[affinity_columns].to_numpy()
    image = axis.imshow(matrix, vmin=0, vmax=1, cmap="viridis", aspect="auto")
    axis.set_xticks(range(len(MONO_CROPS)), MONO_CROPS, rotation=35, ha="right")
    axis.set_yticks(range(len(rows)))
    axis.set_yticklabels(
        [f"{field_uid[:12]}… ({count} px)" for field_uid, count in zip(
            rows["field_uid"], rows["pixel_samples"], strict=True
        )],
        fontsize=7,
    )
    axis.set_title(f"{mixture_label}\nmatched-reference field medians")
    figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
figure.suptitle(
    f"Separately computed, non-compositional affinities — {interpretation_status}",
    color="darkgreen" if reference_gate_passed else "crimson",
)
watermark(figure)
figure.tight_layout()
plt.show()

panel_fields = []
for mixture_label in MIXTURE_PARENTS:
    candidates = (
        field_fingerprints[field_fingerprints["landcover"].eq(mixture_label)]
        .sort_values(["pixel_samples", "field_uid"], ascending=[False, True], kind="stable")
        .head(PIXEL_DNA_FIELDS_PER_MIXTURE)
    )
    panel_fields.extend(candidates["field_uid"].tolist())

n_columns = 2
n_rows = int(np.ceil(len(panel_fields) / n_columns))
figure, axes = plt.subplots(
    n_rows,
    n_columns,
    figsize=(11, 4.6 * n_rows),
    squeeze=False,
)
color_limit = max(
    float(np.abs(mixture_pixel_scores["dual_target_gap"]).max()),
    1e-6,
)
for axis, field_uid in zip(axes.ravel(), panel_fields, strict=False):
    rows = mixture_pixel_scores[mixture_pixel_scores["field_uid"].eq(field_uid)]
    mixture_label = str(rows["landcover"].iloc[0])
    parent_a, parent_b = MIXTURE_PARENTS[mixture_label]
    scatter = axis.scatter(
        rows[crop_to_column[parent_a]],
        rows[crop_to_column[parent_b]],
        c=rows["dual_target_gap"],
        cmap="coolwarm",
        vmin=-color_limit,
        vmax=color_limit,
        s=35,
        alpha=0.85,
        edgecolors="black",
        linewidths=0.25,
    )
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)
    axis.set_xlabel(f"{parent_a} affinity")
    axis.set_ylabel(f"{parent_b} affinity")
    axis.set_title(
        f"{mixture_label}\nfield_uid={field_uid}; every clean pixel (n={len(rows)})",
        fontsize=9,
    )
for axis in axes.ravel()[len(panel_fields):]:
    axis.set_visible(False)
figure.suptitle(
    f"Pixel-level intercropping DNA — {interpretation_status}",
    color="darkgreen" if reference_gate_passed else "crimson",
)
figure.tight_layout(rect=[0.0, 0.0, 0.90, 0.96])
colorbar_axis = figure.add_axes([0.92, 0.15, 0.015, 0.70])
figure.colorbar(
    scatter,
    cax=colorbar_axis,
    label="dual-target gap",
)
watermark(figure)
plt.show()

In [ ]:
field_dna = mixture_pixel_scores.groupby(
    ["field_uid", "landcover", "parent_a", "parent_b"],
    as_index=False,
).agg(
    pixel_samples=("pixel_id", "nunique"),
    median_dual_target_gap=("dual_target_gap", "median"),
    median_parent_contrast=("parent_contrast", "median"),
    positive_gap_pixel_rate=(
        "dual_target_gap",
        lambda values: float((values > 0).mean()),
    ),
    **{
        f"median_{column}": (column, "median")
        for column in affinity_columns
    },
)
field_dna["interpretation"] = interpretation_status
bootstrap_metrics = [
    "median_dual_target_gap",
    "median_parent_contrast",
    "positive_gap_pixel_rate",
    *[f"median_{column}" for column in affinity_columns],
]

bootstrap_rng = np.random.default_rng(RANDOM_SEED + 3)
bootstrap_records = []
for mixture_label in MIXTURE_PARENTS:
    base_fields = field_dna[field_dna["landcover"].eq(mixture_label)]
    base_means = {
        metric: float(base_fields[metric].mean())
        for metric in bootstrap_metrics
    }
    mixture_rows = mixtures[mixtures["landcover"].eq(mixture_label)].copy()
    mixture_field_ids = sorted(mixture_rows["field_uid"].unique())
    reference_pools = {
        crop: mono_reference_fields[
            mono_reference_fields["field_uid"].isin(
                matched_reference_fields[mixture_label][crop]
            )
        ].reset_index(drop=True)
        for crop in MONO_CROPS
    }
    interval_available = len(mixture_field_ids) >= MIN_MIXTURE_FIELDS_FOR_CI
    bootstrap_values = {
        metric: np.empty(N_BOOTSTRAPS, dtype=np.float64)
        for metric in bootstrap_metrics
    }
    if interval_available:
        for bootstrap in range(N_BOOTSTRAPS):
            sampled_reference_parts = []
            for crop in MONO_CROPS:
                pool = reference_pools[crop]
                sampled_positions = bootstrap_rng.integers(
                    0,
                    len(pool),
                    size=len(pool),
                )
                sampled_reference_parts.append(pool.iloc[sampled_positions])
            bootstrap_reference_moments = pd.concat(
                sampled_reference_parts,
                ignore_index=True,
            )
            bootstrap_model = fit_moment_reference(bootstrap_reference_moments)
            bootstrap_scored = attach_mixture_scores(
                mixture_rows,
                bootstrap_model,
                mixture_label,
            )
            bootstrap_fields = bootstrap_scored.groupby(
                "field_uid",
                as_index=False,
            ).agg(
                median_dual_target_gap=("dual_target_gap", "median"),
                median_parent_contrast=("parent_contrast", "median"),
                positive_gap_pixel_rate=(
                    "dual_target_gap",
                    lambda values: float((values > 0).mean()),
                ),
                **{
                    f"median_{column}": (column, "median")
                    for column in affinity_columns
                },
            )
            sampled_field_positions = bootstrap_rng.integers(
                0,
                len(bootstrap_fields),
                size=len(bootstrap_fields),
            )
            sampled_field_summaries = bootstrap_fields.iloc[
                sampled_field_positions
            ]
            for metric in bootstrap_metrics:
                bootstrap_values[metric][bootstrap] = float(
                    sampled_field_summaries[metric].mean()
                )

    for metric in bootstrap_metrics:
        values = bootstrap_values[metric]
        bootstrap_records.append(
            {
                "landcover": mixture_label,
                "metric": metric,
                "field_mean": base_means[metric],
                "ci_low": (
                    float(np.quantile(values, 0.025))
                    if interval_available
                    else np.nan
                ),
                "ci_high": (
                    float(np.quantile(values, 0.975))
                    if interval_available
                    else np.nan
                ),
                "mixture_fields": len(mixture_field_ids),
                "minimum_reference_fields_per_crop": min(
                    len(pool)
                    for pool in matched_reference_fields[mixture_label].values()
                ),
                "reference_pool_sizes": ", ".join(
                    f"{crop}={len(pool)}"
                    for crop, pool in matched_reference_fields[mixture_label].items()
                ),
                "interval_status": (
                    "joint whole-field 95% bootstrap interval"
                    if interval_available
                    else f"suppressed: fewer than {MIN_MIXTURE_FIELDS_FOR_CI} mixture fields"
                ),
                "interpretation": interpretation_status,
            }
        )
bootstrap_summary = pd.DataFrame(bootstrap_records)

colors = {
    "Bean and Maize": "#2ca02c",
    "Irish Potato and Maize": "#9467bd",
}
figure, axis = plt.subplots(figsize=(9, 7))
for label, rows in field_dna.groupby("landcover", sort=False):
    axis.scatter(
        rows["median_parent_contrast"],
        rows["median_dual_target_gap"],
        s=45,
        alpha=0.75,
        color=colors[label],
        label=f"{label} ({len(rows)} fields)",
    )
axis.axhline(0, color="black", linestyle="--", linewidth=1)
axis.axvline(0, color="0.5", linestyle=":", linewidth=1)
axis.set_xlabel("parent contrast: + toward first named parent, − toward second")
axis.set_ylabel(
    "dual-target gap: weaker named-parent affinity − strongest off-target affinity"
)
axis.set_title(f"Field-level intercropping DNA — {interpretation_status}")
axis.legend()
axis.grid(alpha=0.15)
watermark(figure)
figure.tight_layout()
plt.show()

display(bootstrap_summary.set_index(["landcover", "metric"]))
display(
    field_dna.sort_values(
        ["landcover", "median_dual_target_gap"],
        ascending=[True, False],
    )
)

In [ ]:
all_features = np.stack(pixels["_feature"].to_numpy())
pca_mean = all_features.mean(axis=0, keepdims=True)
centered = all_features - pca_mean
_, _, right_vectors = np.linalg.svd(centered, full_matrices=False)
scores = centered @ right_vectors[:2].T
pca_rows = pixels[["field_uid", "pixel_id", "landcover"]].copy()
pca_rows["pca_1"] = scores[:, 0]
pca_rows["pca_2"] = scores[:, 1]
palette = dict(zip(ALL_LABELS, plt.get_cmap("tab10").colors, strict=False))
figure, axis = plt.subplots(figsize=(11, 8))
for label in ALL_LABELS:
    rows = pca_rows[pca_rows["landcover"].eq(label)]
    axis.scatter(
        rows["pca_1"],
        rows["pca_2"],
        s=10,
        alpha=0.45,
        linewidths=0,
        color=palette[label],
        label=f"{label} ({len(rows)} pixels)",
    )
axis.set_xlabel("pixel embedding PC1")
axis.set_ylabel("pixel embedding PC2")
axis.set_title(
    f"{TARGET_WINDOW}: individual pixel embeddings — diagnostic PCA only\n"
    f"{interpretation_status}"
)
axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
axis.grid(alpha=0.15)
watermark(figure)
figure.tight_layout()
plt.show()

## Reading the DNA results

**Parent affinity is a unitless similarity between a TESSERA embedding and geographically matched monocrop reference prototypes in learned representation space. It is not an estimate of crop cover, plant count, biomass, yield, sub-pixel abundance, or physical mixture fraction. A Maize affinity of 0.7 does not mean 70% Maize. Scores depend on the matched reference fields, temporal window, normalization, model, and distance scale, and need not vary linearly with crop composition.**

- The monocrop gate uses whole 5 km UTM spatial blocks, field-level predictions, at least 12 fields and three blocks per parent, per-parent recall and Wilson criteria, plus a 2,000-label-permutation overall test. The overall gate also requires `COMPLETED.json`; a partial pipeline snapshot remains explicitly marked DESCRIPTIVE ONLY even if the internal validation thresholds pass.
- Validation affinities and mixture affinities are not compared numerically. Each mixture cohort searches every eligible clean monocrop field for its geographically matched reference pools, and the candidate counts and nearest-reference distances are reported.
- A positive dual-target gap means both named parents score above every off-target monocrop reference for that pixel or field summary. It is dual-reference similarity, not proof of intercropping.
- Parent contrast indicates which named reference is more similar. Zero means balanced affinity, not equal crop abundance.
- Pixel panels show every sampled 10 m embedding for the named field. Field labels do not prove that both crops occur in each individual pixel, and TESSERA embeddings are nonlinear.
- Nominal intervals are shown only with at least eight mixture fields. They jointly resample whole matched monocrop reference fields, refit the scaler/prototypes, and resample whole mixture fields. Pixels are not treated as independent farms.
- The PCA plot is descriptive only. All affinity calculations use normalized, train-standardized 128-dimensional vectors.
- w4 is the requested 487-day out-of-contract experiment. For a primary in-contract DNA result, rerun with TARGET_WINDOW='w3' and compare conclusions.